# ※ 1단계 > naver open API를 활용하여 네이버지식인에서 각 500개씩 "전주여행"과 "경주여행"을 검색 (특수문자 빼기)((groubby 가능하게)csv백업)-> 
# ※ 2단계 >품사태깅 백업(csv백업), 명사만 추출 -> 빈도분석(DataFrame)(csv백업) -> 빈도 시각화(워드클라우드, Text) -> Word2Vec(csv백업)

'''
conda activate ml-dl-nlp
pip install python-dotenv

'''

# 1. 네이버 open API를 활용하여 검색 추출
- 검색어, no, title, link, description, 명사로추출df(title+' '+description)(total_text)

In [2]:
# pip install python-dotenv (환경변수를 쓰기 위한 라이브러리)
# pip install python-decouple (주피터에서는 사용 안됨)
import os
os.getcwd() # 현재 작업디렉토리

'C:\\ai_x\\source\\07_자연어처리'

In [5]:
%ls .env

 C 드라이브의 볼륨에는 이름이 없습니다.
 볼륨 일련 번호: 3664-591E

 C:\ai_x\source\07_자연어처리 디렉터리

2025-06-18  오후 05:47                58 .env
               1개 파일                  58 바이트
               0개 디렉터리  115,935,191,040 바이트 남음


In [3]:
# 환경변수를 가져오기
from dotenv import load_dotenv
import os
load_dotenv() #dotenv_path='.env' 기본값 생략가능

True

In [4]:
# print(os.getenv('Client_ID'))
# print(os.getenv('Client_Secret'))

bT7ctjQcsaRNLUAu85bK
HyWavVuxdp


In [6]:
# 장고에서는 됨
from decouple import config
# print(config('Client_ID'))
# print(config('Client_Secret'))

bT7ctjQcsaRNLUAu85bK


# 1단계 공부

In [ ]:
# 1단계: 네이버 Open API를 활용하여 지식인 데이터 수집
# 필요한 라이브러리 임포트
import os
import sys
import urllib.request
import urllib.parse
import json
import pandas as pd
import re
from dotenv import load_dotenv
import time

# 환경변수 로드 (.env 파일에서 API 키 정보 가져오기)
load_dotenv()

# 네이버 API 키 설정
client_id = os.getenv('Client_ID')
client_secret = os.getenv('Client_Secret')

def clean_text(text):
    """
    텍스트에서 HTML 태그와 특수문자 제거하는 함수
    Args:
        text (str): 정제할 텍스트
    Returns:
        str: 정제된 텍스트
    """
    if not text:
        return ""
    
    # HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    # 특수문자 제거 (한글, 영문, 숫자, 공백만 남기기)
    text = re.sub(r'[^\w\s가-힣]', ' ', text)
    # 연속된 공백을 하나로 변경
    text = re.sub(r'\s+', ' ', text)
    # 앞뒤 공백 제거
    text = text.strip()
    
    return text

def search_naver_kin(query, display=100, start=1):
    """
    네이버 지식인 검색 함수
    Args:
        query (str): 검색어
        display (int): 한 번에 가져올 결과 수 (최대 100)
        start (int): 검색 시작 위치
    Returns:
        dict: 검색 결과 JSON 데이터
    """
    # 검색어 URL 인코딩
    encText = urllib.parse.quote(query)
    
    # 네이버 지식인 검색 API URL 생성
    url = f"https://openapi.naver.com/v1/search/kin.json?query={encText}&display={display}&start={start}"
    
    # HTTP 요청 객체 생성
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    
    try:
        # API 요청 실행
        response = urllib.request.urlopen(request)
        rescode = response.getcode()
        
        if rescode == 200:
            # 응답 데이터를 JSON으로 파싱
            response_body = response.read()
            return json.loads(response_body.decode('utf-8'))
        else:
            print(f"Error Code: {rescode}")
            return None
            
    except Exception as e:
        print(f"API 요청 중 오류 발생: {e}")
        return None

def collect_kin_data(query, target_count=500):
    """
    네이버 지식인에서 지정된 개수만큼 데이터 수집
    Args:
        query (str): 검색어
        target_count (int): 수집할 데이터 개수
    Returns:
        list: 수집된 데이터 리스트
    """
    all_items = []
    start = 1
    display = 100  # 한 번에 최대 100개까지 가져올 수 있음
    
    print(f"'{query}' 검색 시작...")
    
    while len(all_items) < target_count:
        # 남은 개수가 100개 미만이면 display 조정
        remaining = target_count - len(all_items)
        current_display = min(display, remaining)
        
        print(f"진행상황: {len(all_items)}/{target_count} - 현재 요청: {start}~{start + current_display - 1}")
        
        # API 호출
        result = search_naver_kin(query, current_display, start)
        
        if result and 'items' in result:
            items = result['items']
            
            if not items:  # 더 이상 검색 결과가 없으면 중단
                print("더 이상 검색 결과가 없습니다.")
                break
                
            all_items.extend(items)
            start += current_display
            
            # API 호출 제한을 고려하여 잠시 대기
            time.sleep(0.1)
        else:
            print("API 요청 실패")
            break
    
    print(f"'{query}' 검색 완료: {len(all_items)}개 수집")
    return all_items[:target_count]  # 정확히 target_count 개수만 반환

def process_data(items, search_query):
    """
    수집된 데이터를 DataFrame으로 변환하고 정제
    Args:
        items (list): 검색 결과 아이템 리스트
        search_query (str): 검색어
    Returns:
        pandas.DataFrame: 정제된 데이터프레임
    """
    processed_data = []
    
    for idx, item in enumerate(items, 1):
        # 제목과 설명에서 HTML 태그 및 특수문자 제거
        title_clean = clean_text(item.get('title', ''))
        description_clean = clean_text(item.get('description', ''))
        
        # 제목과 설명을 합쳐서 전체 텍스트 생성
        total_text = f"{title_clean} {description_clean}".strip()
        
        processed_data.append({
            '검색어': search_query,
            'no': idx,
            'title': title_clean,
            'link': item.get('link', ''),
            'description': description_clean,
            'total_text': total_text
        })
    
    return pd.DataFrame(processed_data)

def main():
    """
    메인 실행 함수
    """
    # API 키 확인
    if not client_id or not client_secret:
        print("오류: 네이버 API 키가 설정되지 않았습니다.")
        print(".env 파일에 Client_ID와 Client_Secret을 설정해주세요.")
        return
    
    # 검색어 설정
    search_queries = ["전주여행", "경주여행"]
    target_count = 500  # 각 검색어당 수집할 데이터 개수
    
    all_dataframes = []
    
    # 각 검색어별로 데이터 수집
    for query in search_queries:
        print(f"\n=== {query} 데이터 수집 시작 ===")
        
        # 네이버 지식인에서 데이터 수집
        items = collect_kin_data(query, target_count)
        
        if items:
            # 데이터 정제 및 DataFrame 변환
            df = process_data(items, query)
            all_dataframes.append(df)
            
            # 개별 CSV 파일로 저장
            filename = f"naver_kin_{query}_raw.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            print(f"{query} 데이터 저장 완료: {filename}")
            
            # 데이터 미리보기
            print(f"\n{query} 데이터 미리보기:")
            print(df.head(3))
            print(f"총 {len(df)}개 데이터 수집 완료\n")
        else:
            print(f"{query} 데이터 수집 실패")
    
    # 전체 데이터 통합
    if all_dataframes:
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        
        # 통합 CSV 파일로 저장
        combined_filename = "naver_kin_combined_raw.csv"
        combined_df.to_csv(combined_filename, index=False, encoding='utf-8-sig')
        print(f"\n전체 통합 데이터 저장 완료: {combined_filename}")
        
        # 전체 데이터 요약 정보
        print(f"\n=== 데이터 수집 완료 ===")
        print(f"전체 데이터 개수: {len(combined_df)}")
        print("\n검색어별 데이터 개수:")
        print(combined_df['검색어'].value_counts())
        
        # 데이터 구조 확인
        print(f"\n데이터 구조:")
        print(combined_df.info())
        
        return combined_df
    else:
        print("데이터 수집에 실패했습니다.")
        return None

# 실행
if __name__ == "__main__":
    df = main()

# 2단계 공부

In [ ]:
# 2단계: 품사태깅, 명사추출, 빈도분석, 시각화, Word2Vec
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from wordcloud import WordCloud
import matplotlib.font_manager as fm

# 자연어 처리 라이브러리
from konlpy.tag import Okt, Komoran, Kkma
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer

# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['axes.unicode_minus'] = False

class TextAnalyzer:
    """
    텍스트 분석을 위한 클래스
    """
    def __init__(self):
        """
        초기화: 형태소 분석기 설정
        """
        self.okt = Okt()  # Open Korean Text (빠르고 정확도 적당)
        self.komoran = Komoran()  # 정확도 높음
        self.kkma = Kkma()  # 가장 정확하지만 느림
        
        # 불용어 설정 (의미없는 단어들)
        self.stop_words = {
            '이', '그', '저', '것', '수', '등', '및', '또한', '하지만', '그러나', '따라서',
            '때문', '위해', '통해', '대해', '관해', '에서', '에게', '에도', '부터', '까지',
            '정말', '진짜', '너무', '매우', '아주', '좀', '조금', '많이', '약간',
            '있다', '없다', '하다', '되다', '이다', '아니다', '같다', '다르다',
            '좋다', '나쁘다', '크다', '작다', '높다', '낮다', '길다', '짧다'
        }
    
    def pos_tagging(self, text, analyzer='okt'):
        """
        품사 태깅 함수
        Args:
            text (str): 분석할 텍스트
            analyzer (str): 사용할 형태소 분석기 ('okt', 'komoran', 'kkma')
        Returns:
            list: 품사 태깅 결과 [(단어, 품사), ...]
        """
        if not text or pd.isna(text):
            return []
        
        try:
            if analyzer == 'okt':
                return self.okt.pos(text)
            elif analyzer == 'komoran':
                return self.komoran.pos(text)
            elif analyzer == 'kkma':
                return self.kkma.pos(text)
            else:
                return self.okt.pos(text)
        except:
            return []
    
    def extract_nouns(self, text, analyzer='okt', min_length=2):
        """
        명사만 추출하는 함수
        Args:
            text (str): 분석할 텍스트
            analyzer (str): 사용할 형태소 분석기
            min_length (int): 최소 단어 길이
        Returns:
            list: 추출된 명사 리스트
        """
        if not text or pd.isna(text):
            return []
        
        try:
            if analyzer == 'okt':
                nouns = self.okt.nouns(text)
            elif analyzer == 'komoran':
                pos_tags = self.komoran.pos(text)
                nouns = [word for word, pos in pos_tags if pos.startswith('N')]
            elif analyzer == 'kkma':
                pos_tags = self.kkma.pos(text)
                nouns = [word for word, pos in pos_tags if pos.startswith('N')]
            else:
                nouns = self.okt.nouns(text)
            
            # 필터링: 길이 조건, 불용어 제거, 숫자만 있는 단어 제거
            filtered_nouns = []
            for noun in nouns:
                if (len(noun) >= min_length and 
                    noun not in self.stop_words and 
                    not noun.isdigit() and 
                    not re.match(r'^[a-zA-Z]+$', noun)):  # 영어 단어 제외
                    filtered_nouns.append(noun)
            
            return filtered_nouns
        except:
            return []

def process_pos_tagging(df, text_column='total_text'):
    """
    데이터프레임의 텍스트에 품사 태깅 적용
    Args:
        df (DataFrame): 처리할 데이터프레임
        text_column (str): 텍스트가 있는 컬럼명
    Returns:
        DataFrame: 품사 태깅 결과가 추가된 데이터프레임
    """
    analyzer = TextAnalyzer()
    
    print("품사 태깅 진행 중...")
    
    # 품사 태깅 결과 저장
    pos_results = []
    noun_results = []
    
    for idx, text in enumerate(df[text_column]):
        if idx % 100 == 0:
            print(f"진행상황: {idx}/{len(df)}")
        
        # 품사 태깅
        pos_tags = analyzer.pos_tagging(text)
        pos_results.append(pos_tags)
        
        # 명사 추출
        nouns = analyzer.extract_nouns(text)
        noun_results.append(nouns)
    
    # 데이터프레임에 결과 추가
    df_copy = df.copy()
    df_copy['pos_tags'] = pos_results
    df_copy['nouns'] = noun_results
    df_copy['nouns_text'] = df_copy['nouns'].apply(lambda x: ' '.join(x) if x else '')
    
    print("품사 태깅 완료!")
    return df_copy

def frequency_analysis(df, search_query=None):
    """
    명사 빈도 분석
    Args:
        df (DataFrame): 분석할 데이터프레임
        search_query (str): 특정 검색어로 필터링 (None이면 전체)
    Returns:
        DataFrame: 빈도 분석 결과
    """
    # 특정 검색어로 필터링
    if search_query:
        df_filtered = df[df['검색어'] == search_query].copy()
        print(f"'{search_query}' 데이터로 빈도 분석 진행...")
    else:
        df_filtered = df.copy()
        print("전체 데이터로 빈도 분석 진행...")
    
    # 모든 명사를 하나의 리스트로 합치기
    all_nouns = []
    for nouns_list in df_filtered['nouns']:
        if nouns_list:
            all_nouns.extend(nouns_list)
    
    # 빈도 계산
    noun_counter = Counter(all_nouns)
    
    # 빈도 상위 100개 추출
    top_nouns = noun_counter.most_common(100)
    
    # DataFrame으로 변환
    freq_df = pd.DataFrame(top_nouns, columns=['단어', '빈도'])
    freq_df['순위'] = range(1, len(freq_df) + 1)
    freq_df['검색어'] = search_query if search_query else '전체'
    
    print(f"총 {len(all_nouns)}개 명사 중 상위 100개 추출 완료")
    
    return freq_df

def visualize_frequency(freq_df, title="단어 빈도 분석", top_n=20):
    """
    빈도 분석 결과 시각화
    Args:
        freq_df (DataFrame): 빈도 분석 결과
        title (str): 그래프 제목
        top_n (int): 상위 몇 개까지 표시할지
    """
    plt.figure(figsize=(12, 8))
    
    # 상위 n개 데이터 선택
    top_data = freq_df.head(top_n)
    
    # 막대 그래프
    plt.subplot(2, 1, 1)
    bars = plt.bar(range(len(top_data)), top_data['빈도'], color='skyblue', alpha=0.7)
    plt.title(f'{title} - 상위 {top_n}개 단어', fontsize=14, fontweight='bold')
    plt.xlabel('단어', fontsize=12)
    plt.ylabel('빈도', fontsize=12)
    plt.xticks(range(len(top_data)), top_data['단어'], rotation=45, ha='right')
    
    # 막대 위에 빈도 수 표시
    for i, bar in enumerate(bars):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{int(height)}', ha='center', va='bottom', fontsize=10)
    
    # 워드 클라우드
    plt.subplot(2, 1, 2)
    
    # 워드클라우드용 텍스트 생성
    word_freq_dict = dict(zip(freq_df['단어'], freq_df['빈도']))
    
    # 워드클라우드 생성
    wordcloud = WordCloud(
        font_path='malgun.ttf',  # Windows 한글 폰트
        # font_path='/System/Library/Fonts/AppleGothic.ttf',  # Mac 한글 폰트
        width=800, 
        height=400,
        background_color='white',
        max_words=50,
        colormap='viridis'
    ).generate_from_frequencies(word_freq_dict)
    
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('워드 클라우드', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def create_word2vec_model(df, vector_size=100, window=5, min_count=2):
    """
    Word2Vec 모델 생성
    Args:
        df (DataFrame): 학습할 데이터프레임
        vector_size (int): 벡터 차원 수
        window (int): 컨텍스트 윈도우 크기
        min_count (int): 최소 단어 출현 빈도
    Returns:
        Word2Vec: 훈련된 Word2Vec 모델
    """
    print("Word2Vec 모델 학습 시작...")
    
    # 학습 데이터 준비 (각 문서의 명사 리스트)
    sentences = []
    for nouns_list in df['nouns']:
        if nouns_list and len(nouns_list) > 1:  # 명사가 2개 이상인 경우만
            sentences.append(nouns_list)
    
    print(f"학습 문장 수: {len(sentences)}")
    
    # Word2Vec 모델 학습
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,  # 벡터 차원
        window=window,           # 컨텍스트 윈도우
        min_count=min_count,     # 최소 출현 빈도
        workers=4,               # 병렬 처리 워커 수
        sg=0,                    # CBOW 사용 (1이면 Skip-gram)
        epochs=100               # 학습 에포크
    )
    
    print("Word2Vec 모델 학습 완료!")
    print(f"어휘 크기: {len(model.wv.key_to_index)}")
    
    return model

def analyze_word2vec(model, target_words=['여행', '관광', '맛집']):
    """
    Word2Vec 모델 분석
    Args:
        model (Word2Vec): 훈련된 Word2Vec 모델
        target_words (list): 분석할 대상 단어들
    Returns:
        dict: 단어별 유사도 분석 결과
    """
    results = {}
    
    print("Word2Vec 모델 분석 중...")
    
    for word in target_words:
        if word in model.wv.key_to_index:
            # 가장 유사한 단어 10개 찾기
            similar_words = model.wv.most_similar(word, topn=10)
            results[word] = similar_words
            
            print(f"\n'{word}'와 유사한 단어들:")
            for similar_word, similarity in similar_words:
                print(f"  {similar_word}: {similarity:.4f}")
        else:
            print(f"'{word}'는 모델 어휘에 없습니다.")
            results[word] = []
    
    return results

def save_word2vec_results(model, filename_prefix="word2vec"):
    """
    Word2Vec 결과를 CSV로 저장
    Args:
        model (Word2Vec): 훈련된 Word2Vec 모델
        filename_prefix (str): 파일명 접두사
    """
    # 모든 단어와 벡터 추출
    word_vectors = []
    
    for word in model.wv.key_to_index:
        vector = model.wv[word]
        word_data = {'단어': word}
        
        # 벡터의 각 차원을 컬럼으로 추가
        for i, val in enumerate(vector):
            word_data[f'dim_{i}'] = val
        
        word_vectors.append(word_data)
    
    # DataFrame으로 변환 후 저장
    vectors_df = pd.DataFrame(word_vectors)
    vectors_filename = f"{filename_prefix}_vectors.csv"
    vectors_df.to_csv(vectors_filename, index=False, encoding='utf-8-sig')
    
    print(f"Word2Vec 벡터 저장 완료: {vectors_filename}")

def main_step2():
    """
    2단계 메인 실행 함수
    """
    try:
        # 1단계에서 생성된 CSV 파일 로드
        print("1단계 데이터 로드 중...")
        df = pd.read_csv("naver_kin_combined_raw.csv", encoding='utf-8-sig')
        print(f"로드된 데이터 수: {len(df)}")
        
        # 2-1. 품사 태깅 및 명사 추출
        print("\n=== 2-1. 품사 태깅 및 명사 추출 ===")
        df_tagged = process_pos_tagging(df)
        
        # 품사 태깅 결과 저장
        tagged_filename = "naver_kin_pos_tagged.csv"
        df_tagged.to_csv(tagged_filename, index=False, encoding='utf-8-sig')
        print(f"품사 태깅 결과 저장: {tagged_filename}")
        
        # 2-2. 빈도 분석
        print("\n=== 2-2. 빈도 분석 ===")
        
        # 전체 빈도 분석
        freq_all = frequency_analysis(df_tagged)
        freq_all.to_csv("frequency_analysis_all.csv", index=False, encoding='utf-8-sig')
        
        # 검색어별 빈도 분석
        freq_jeonju = frequency_analysis(df_tagged, "전주여행")
        freq_jeonju.to_csv("frequency_analysis_jeonju.csv", index=False, encoding='utf-8-sig')
        
        freq_gyeongju = frequency_analysis(df_tagged, "경주여행")
        freq_gyeongju.to_csv("frequency_analysis_gyeongju.csv", index=False, encoding='utf-8-sig')
        
        print("빈도 분석 결과 저장 완료")
        
        # 2-3. 시각화
        print("\n=== 2-3. 빈도 분석 시각화 ===")
        visualize_frequency(freq_all, "전체 데이터 단어 빈도 분석")
        visualize_frequency(freq_jeonju, "전주여행 단어 빈도 분석")
        visualize_frequency(freq_gyeongju, "경주여행 단어 빈도 분석")
        
        # 2-4. Word2Vec 모델 학습
        print("\n=== 2-4. Word2Vec 모델 학습 ===")
        w2v_model = create_word2vec_model(df_tagged)
        
        # Word2Vec 모델 저장
        w2v_model.save("word2vec_model.model")
        print("Word2Vec 모델 저장 완료: word2vec_model.model")
        
        # Word2Vec 분석
        w2v_results = analyze_word2vec(w2v_model, ['여행', '관광', '맛집', '전주', '경주', '한옥'])
        
        # Word2Vec 결과를 CSV로 저장
        save_word2vec_results(w2v_model)
        
        print("\n=== 2단계 완료 ===")
        print("생성된 파일들:")
        print("- naver_kin_pos_tagged.csv: 품사 태깅 결과")
        print("- frequency_analysis_all.csv: 전체 빈도 분석")
        print("- frequency_analysis_jeonju.csv: 전주여행 빈도 분석")
        print("- frequency_analysis_gyeongju.csv: 경주여행 빈도 분석")
        print("- word2vec_model.model: Word2Vec 모델")
        print("- word2vec_vectors.csv: Word2Vec 벡터 데이터")
        
        return df_tagged, freq_all, freq_jeonju, freq_gyeongju, w2v_model
        
    except FileNotFoundError:
        print("오류: 1단계에서 생성된 CSV 파일을 찾을 수 없습니다.")
        print("먼저 1단계를 실행해주세요.")
        return None
    except Exception as e:
        print(f"오류 발생: {e}")
        return None

# 실행
if __name__ == "__main__":
    results = main_step2()

# 3 합친코드 공부

In [ ]:
# 전체 프로젝트 실행 스크립트
# 네이버 Open API를 활용한 여행 관련 텍스트 분석 프로젝트

"""
프로젝트 개요:
1단계: 네이버 지식인에서 "전주여행", "경주여행" 각 500개씩 데이터 수집
2단계: 품사태깅, 명사추출, 빈도분석, 시각화, Word2Vec 모델링

필요한 라이브러리 설치:
pip install pandas numpy matplotlib seaborn wordcloud konlpy gensim scikit-learn python-dotenv

환경 설정:
.env 파일에 네이버 API 키 설정 필요
Client_ID=your_client_id
Client_Secret=your_client_secret
"""

import os
import sys
import urllib.request
import urllib.parse
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import time
from wordcloud import WordCloud
import matplotlib.font_manager as fm
from dotenv import load_dotenv

# 자연어 처리 라이브러리
from konlpy.tag import Okt, Komoran, Kkma
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['axes.unicode_minus'] = False

class NaverAPICollector:
    """
    네이버 API를 활용한 데이터 수집 클래스
    """
    def __init__(self):
        # 환경변수 로드
        load_dotenv()
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        if not self.client_id or not self.client_secret:
            raise ValueError("네이버 API 키가 설정되지 않았습니다. .env 파일을 확인해주세요.")
    
    def clean_text(self, text):
        """텍스트 정제 함수"""
        if not text:
            return ""
        
        # HTML 태그 제거
        text = re.sub(r'<[^>]+>', '', text)
        # 특수문자 제거 (한글, 영문, 숫자, 공백만 남기기)
        text = re.sub(r'[^\w\s가-힣]', ' ', text)
        # 연속된 공백을 하나로 변경
        text = re.sub(r'\s+', ' ', text)
        # 앞뒤 공백 제거
        text = text.strip()
        
        return text
    
    def search_naver_kin(self, query, display=100, start=1):
        """네이버 지식인 검색 함수"""
        encText = urllib.parse.quote(query)
        url = f"https://openapi.naver.com/v1/search/kin.json?query={encText}&display={display}&start={start}"
        
        request = urllib.request.Request(url)
        request.add_header("X-Naver-Client-Id", self.client_id)
        request.add_header("X-Naver-Client-Secret", self.client_secret)
        
        try:
            response = urllib.request.urlopen(request)
            rescode = response.getcode()
            
            if rescode == 200:
                response_body = response.read()
                return json.loads(response_body.decode('utf-8'))
            else:
                print(f"Error Code: {rescode}")
                return None
        except Exception as e:
            print(f"API 요청 중 오류 발생: {e}")
            return None
    
    def collect_data(self, query, target_count=500):
        """지정된 개수만큼 데이터 수집"""
        all_items = []
        start = 1
        display = 100
        
        print(f"'{query}' 검색 시작...")
        
        while len(all_items) < target_count:
            remaining = target_count - len(all_items)
            current_display = min(display, remaining)
            
            print(f"진행상황: {len(all_items)}/{target_count}")
            
            result = self.search_naver_kin(query, current_display, start)
            
            if result and 'items' in result:
                items = result['items']
                if not items:
                    print("더 이상 검색 결과가 없습니다.")
                    break
                
                all_items.extend(items)
                start += current_display
                time.sleep(0.1)  # API 호출 제한 고려
            else:
                print("API 요청 실패")
                break
        
        print(f"'{query}' 검색 완료: {len(all_items)}개 수집")
        return all_items[:target_count]
    
    def process_data(self, items, search_query):
        """수집된 데이터를 DataFrame으로 변환"""
        processed_data = []
        
        for idx, item in enumerate(items, 1):
            title_clean = self.clean_text(item.get('title', ''))
            description_clean = self.clean_text(item.get('description', ''))
            total_text = f"{title_clean} {description_clean}".strip()
            
            processed_data.append({
                '검색어': search_query,
                'no': idx,
                'title': title_clean,
                'link': item.get('link', ''),
                'description': description_clean,
                'total_text': total_text
            })
        
        return pd.DataFrame(processed_data)

class TextAnalyzer:
    """
    텍스트 분석 클래스
    """
    def __init__(self):
        self.okt = Okt()
        self.komoran = Komoran()
        
        # 불용어 설정
        self.stop_words = {
            '이', '그', '저', '것', '수', '등', '및', '또한', '하지만', '그러나', '따라서',
            '때문', '위해', '통해', '대해', '관해', '에서', '에게', '에도', '부터', '까지',
            '정말', '진짜', '너무', '매우', '아주', '좀', '조금', '많이', '약간',
            '있다', '없다', '하다', '되다', '이다', '아니다', '같다', '다르다',
            '좋다', '나쁘다', '크다', '작다', '높다', '낮다', '길다', '짧다'
        }
    
    def extract_nouns(self, text, min_length=2):
        """명사 추출 함수"""
        if not text or pd.isna(text):
            return []
        
        try:
            nouns = self.okt.nouns(text)
            
            filtered_nouns = []
            for noun in nouns:
                if (len(noun) >= min_length and 
                    noun not in self.stop_words and 
                    not noun.isdigit() and 
                    not re.match(r'^[a-zA-Z]+, noun)):
                    filtered_nouns.append(noun)
            
            return filtered_nouns
        except:
            return []
    
    def pos_tagging(self, text):
        """품사 태깅 함수"""
        if not text or pd.isna(text):
            return []
        
        try:
            return self.okt.pos(text)
        except:
            return []

class DataVisualizer:
    """
    데이터 시각화 클래스
    """
    @staticmethod
    def plot_frequency(freq_df, title="단어 빈도 분석", top_n=20):
        """빈도 분석 결과 시각화"""
        plt.figure(figsize=(15, 10))
        
        # 상위 n개 데이터
        top_data = freq_df.head(top_n)
        
        # 막대 그래프
        plt.subplot(2, 1, 1)
        bars = plt.bar(range(len(top_data)), top_data['빈도'], 
                      color='skyblue', alpha=0.7, edgecolor='navy')
        plt.title(f'{title} - 상위 {top_n}개 단어', fontsize=16, fontweight='bold')
        plt.xlabel('단어', fontsize=12)
        plt.ylabel('빈도', fontsize=12)
        plt.xticks(range(len(top_data)), top_data['단어'], rotation=45, ha='right')
        
        # 막대 위에 빈도 수 표시
        for i, bar in enumerate(bars):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{int(height)}', ha='center', va='bottom', fontsize=9)
        
        plt.grid(axis='y', alpha=0.3)
        
        # 워드 클라우드
        plt.subplot(2, 1, 2)
        word_freq_dict = dict(zip(freq_df['단어'], freq_df['빈도']))
        
        try:
            wordcloud = WordCloud(
                font_path='malgun.ttf',  # Windows
                width=1000, 
                height=500,
                background_color='white',
                max_words=100,
                colormap='viridis',
                relative_scaling=0.5
            ).generate_from_frequencies(word_freq_dict)
            
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')
            plt.title('워드 클라우드', fontsize=16, fontweight='bold')
        except Exception as e:
            plt.text(0.5, 0.5, f'워드클라우드 생성 실패\n{str(e)}', 
                    ha='center', va='center', fontsize=12, transform=plt.gca().transAxes)
            plt.axis('off')
        
        plt.tight_layout()
        plt.savefig(f'{title.replace(" ", "_")}_visualization.png', dpi=300, bbox_inches='tight')
        plt.show()

def step1_data_collection():
    """1단계: 데이터 수집"""
    print("="*50)
    print("1단계: 네이버 지식인 데이터 수집 시작")
    print("="*50)
    
    try:
        collector = NaverAPICollector()
        search_queries = ["전주여행", "경주여행"]
        target_count = 500
        
        all_dataframes = []
        
        for query in search_queries:
            print(f"\n{query} 데이터 수집 중...")
            items = collector.collect_data(query, target_count)
            
            if items:
                df = collector.process_data(items, query)
                all_dataframes.append(df)
                
                # 개별 저장
                filename = f"naver_kin_{query}_raw.csv"
                df.to_csv(filename, index=False, encoding='utf-8-sig')
                print(f"저장 완료: {filename}")
        
        # 통합 데이터 저장
        if all_dataframes:
            combined_df = pd.concat(all_dataframes, ignore_index=True)
            combined_df.to_csv("naver_kin_combined_raw.csv", index=False, encoding='utf-8-sig')
            
            print(f"\n1단계 완료!")
            print(f"총 데이터 개수: {len(combined_df)}")
            print("검색어별 데이터 개수:")
            print(combined_df['검색어'].value_counts())
            
            return combined_df
        else:
            print("데이터 수집 실패")
            return None
            
    except Exception as e:
        print(f"1단계 오류: {e}")
        return None

def step2_text_analysis():
    """2단계: 텍스트 분석"""
    print("="*50)
    print("2단계: 텍스트 분석 시작")
    print("="*50)
    
    try:
        # 데이터 로드
        df = pd.read_csv("naver_kin_combined_raw.csv", encoding='utf-8-sig')
        print(f"데이터 로드 완료: {len(df)}개")
        
        analyzer = TextAnalyzer()
        visualizer = DataVisualizer()
        
        # 2-1. 품사 태깅 및 명사 추출
        print("\n2-1. 품사 태깅 및 명사 추출...")
        pos_results = []
        noun_results = []
        
        for idx, text in enumerate(df['total_text']):
            if idx % 100 == 0:
                print(f"진행상황: {idx}/{len(df)}")
            
            pos_tags = analyzer.pos_tagging(text)
            nouns = analyzer.extract_nouns(text)
            
            pos_results.append(pos_tags)
            noun_results.append(nouns)
        
        df['pos_tags'] = pos_results
        df['nouns'] = noun_results
        df['nouns_text'] = df['nouns'].apply(lambda x: ' '.join(x) if x else '')
        
        # 품사 태깅 결과 저장
        df.to_csv("naver_kin_pos_tagged.csv", index=False, encoding='utf-8-sig')
        print("품사 태깅 결과 저장 완료")
        
        # 2-2. 빈도 분석
        print("\n2-2. 빈도 분석...")
        
        def frequency_analysis(df_input, search_query=None):
            if search_query:
                df_filtered = df_input[df_input['검색어'] == search_query]
            else:
                df_filtered = df_input
            
            all_nouns = []
            for nouns_list in df_filtered['nouns']:
                if nouns_list:
                    all_nouns.extend(nouns_list)
            
            noun_counter = Counter(all_nouns)
            top_nouns = noun_counter.most_common(100)
            
            freq_df = pd.DataFrame(top_nouns, columns=['단어', '빈도'])
            freq_df['순위'] = range(1, len(freq_df) + 1)
            freq_df['검색어'] = search_query if search_query else '전체'
            
            return freq_df
        
        # 전체 및 검색어별 빈도 분석
        freq_all = frequency_analysis(df)
        freq_jeonju = frequency_analysis(df, "전주여행")
        freq_gyeongju = frequency_analysis(df, "경주여행")
        
        # 빈도 분석 결과 저장
        freq_all.to_csv("frequency_analysis_all.csv", index=False, encoding='utf-8-sig')
        freq_jeonju.to_csv("frequency_analysis_jeonju.csv", index=False, encoding='utf-8-sig')
        freq_gyeongju.to_csv("frequency_analysis_gyeongju.csv", index=False, encoding='utf-8-sig')
        print("빈도 분석 결과 저장 완료")
        
        # 2-3. 시각화
        print("\n2-3. 시각화...")
        visualizer.plot_frequency(freq_all, "전체 데이터 단어 빈도 분석")
        visualizer.plot_frequency(freq_jeonju, "전주여행 단어 빈도 분석")
        visualizer.plot_frequency(freq_gyeongju, "경주여행 단어 빈도 분석")
        
        # 2-4. Word2Vec 모델링
        print("\n2-4. Word2Vec 모델 학습...")
        
        # 학습 데이터 준비
        sentences = []
        for nouns_list in df['nouns']:
            if nouns_list and len(nouns_list) > 1:
                sentences.append(nouns_list)
        
        print(f"학습 문장 수: {len(sentences)}")
        
        # Word2Vec 모델 학습
        model = Word2Vec(
            sentences=sentences,
            vector_size=100,
            window=5,
            min_count=2,
            workers=4,
            sg=0,
            epochs=100
        )
        
        # 모델 저장
        model.save("word2vec_model.model")
        print("Word2Vec 모델 저장 완료")
        
        # Word2Vec 분석
        target_words = ['여행', '관광', '맛집', '전주', '경주', '한옥']
        print("\nWord2Vec 유사도 분석:")
        
        for word in target_words:
            if word in model.wv.key_to_index:
                similar_words = model.wv.most_similar(word, topn=5)
                print(f"\n'{word}'와 유사한 단어들:")
                for similar_word, similarity in similar_words:
                    print(f"  {similar_word}: {similarity:.4f}")
        
        # Word2Vec 벡터 저장
        word_vectors = []
        for word in model.wv.key_to_index:
            vector = model.wv[word]
            word_data = {'단어': word}
            for i, val in enumerate(vector):
                word_data[f'dim_{i}'] = val
            word_vectors.append(word_data)
        
        vectors_df = pd.DataFrame(word_vectors)
        vectors_df.to_csv("word2vec_vectors.csv", index=False, encoding='utf-8-sig')
        print("Word2Vec 벡터 저장 완료")
        
        print("\n2단계 완료!")
        return df, freq_all, freq_jeonju, freq_gyeongju, model
        
    except Exception as e:
        print(f"2단계 오류: {e}")
        return None

def main():
    """메인 실행 함수"""
    print("네이버 Open API 텍스트 분석 프로젝트 시작")
    print("필요한 파일: .env (네이버 API 키 포함)")
    print("필요한 라이브러리: pandas, numpy, matplotlib, seaborn, wordcloud, konlpy, gensim")
    
    # 실행할 단계 선택
    choice = input("\n실행할 단계를 선택하세요 (1: 1단계만, 2: 2단계만, 3: 전체): ")
    
    if choice == "1":
        df = step1_data_collection()
        
    elif choice == "2":
        if not os.path.exists("naver_kin_combined_raw.csv"):
            print("1단계 데이터 파일이 없습니다. 먼저 1단계를 실행해주세요.")
            return
        results = step2_text_analysis()
        
    elif choice == "3":
        # 전체 실행
        df = step1_data_collection()
        if df is not None:
            results = step2_text_analysis()
        
    else:
        print("잘못된 선택입니다.")
    
    print("\n프로젝트 완료!")
    print("\n생성된 파일들:")
    files = [
        "naver_kin_combined_raw.csv",
        "naver_kin_pos_tagged.csv", 
        "frequency_analysis_all.csv",
        "frequency_analysis_jeonju.csv",
        "frequency_analysis_gyeongju.csv",
        "word2vec_model.model",
        "word2vec_vectors.csv"
    ]
    
    for file in files:
        if os.path.exists(file):
            print(f"✓ {file}")

if __name__ == "__main__":
    main()

# 4. 내용 공부

In [ ]:
'''
CSV 파일들:

naver_kin_combined_raw.csv: 원본 데이터
naver_kin_pos_tagged.csv: 품사태깅 결과
frequency_analysis_*.csv: 빈도분석 결과
word2vec_vectors.csv: Word2Vec 벡터

시각화: 막대그래프 + 워드클라우드 이미지 파일들

모델 파일: word2vec_model.model: 훈련된 Word2Vec 모델
        
데이터 수집: 각 검색어당 500개씩 총 1,000개 데이터
텍스트 정제: HTML 태그, 특수문자 제거
품사 태깅: KoNLPy의 Okt 사용
명사 추출: 불용어 제거, 길이 필터링
빈도 분석: 전체/검색어별 상위 100개 단어
시각화: 막대그래프 + 워드클라우드
Word2Vec: 단어 임베딩 모델 학습
'''